# 13 — Ridge Classifier

**Algorithm:** Ridge Classifier (L2-penalised linear model via regression)  
**Features:** TF-IDF (50K unigrams + bigrams)  
Ridge Classifier solves a multiclass ridge regression problem and predicts the class with highest score. It is faster than logistic regression and often competitive.

In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import (setup_paths, evaluate_and_print, plot_confusion_matrix,
                   plot_roc_curve, plot_pr_curve, plot_learning_curve)
from sklearn.metrics import accuracy_score
BASE_DIR, FIGURES_DIR = setup_paths()

In [ ]:
from preprocess import load_all_datasets, get_train_test_split
df = load_all_datasets()
X_train, X_test, y_train, y_test = get_train_test_split(df)
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True, min_df=2, max_df=0.95)
X_tr = tfidf.fit_transform(X_train)
X_te = tfidf.transform(X_test)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

In [ ]:
from sklearn.linear_model import RidgeClassifier
MODEL_NAME = 'Ridge Classifier'
model = RidgeClassifier()
t0 = time.time()
model.fit(X_tr, y_train)
train_time = time.time() - t0
print(f'Training completed in {train_time:.2f}s')

In [ ]:
metrics, y_pred, y_prob = evaluate_and_print(model, X_te, y_test, MODEL_NAME, train_time)

In [ ]:
plot_confusion_matrix(y_test, y_pred, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_roc_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_pr_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

## Regularisation Path (Alpha vs Accuracy)

In [ ]:
alphas = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]
tr_accs, te_accs = [], []
for a in alphas:
    m = RidgeClassifier(alpha=a)
    m.fit(X_tr, y_train)
    tr_accs.append(accuracy_score(y_train, m.predict(X_tr)))
    te_accs.append(accuracy_score(y_test, m.predict(X_te)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(alphas, tr_accs, 'o-', color='#3498db', lw=2, ms=7, label='Train Accuracy')
ax.semilogx(alphas, te_accs, 's-', color='#e74c3c', lw=2, ms=7, label='Test Accuracy')
ax.set_xlabel('Alpha (regularisation, log scale)')
ax.set_ylabel('Accuracy')
ax.set_title('Ridge Classifier — Regularisation Path')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'ridge_reg_path.png'), dpi=150, bbox_inches='tight')
plt.show()

## Learning Curve

In [ ]:
plot_learning_curve(model, X_tr, y_train, X_te, y_test, MODEL_NAME, FIGURES_DIR)